In [ ]:
import json
import re
from datetime import datetime
from sentence_transformers import SentenceTransformer
import umap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.cluster import KMeans


# --- Load and extract user messages + timestamps from JSON ---
with open("conversations.json", "r", encoding="utf-8") as f:
    data = json.load(f)

user_messages = []
user_timestamps = []

for convo in data:
    mapping = convo.get("mapping", {})
    for msg in mapping.values():
        message = msg.get("message")
        if not message:
            continue
        author = message.get("author", {})
        if author.get("role") == "user":
            parts = message.get("content", {}).get("parts", [])
            timestamp = message.get("create_time")
            if parts and isinstance(parts[0], str):
                user_messages.append(parts[0])
                user_timestamps.append(timestamp)

# --- First-stage filter: short, duplicates, empty ---
raw_pairs = list(zip(user_messages, user_timestamps))

# Get unique pairs by converting to a set, then sort deterministically by timestamp
unique_pairs = sorted(list(set(raw_pairs)), key=lambda item: item[1])

# Now apply your filters
filtered_pairs = [
    (msg, ts) for msg, ts in unique_pairs
    if len(msg.split()) >= 7 and re.search(r'\w', msg)
]

# --- Second-stage filter: noisy/loggy/cody lines ---
def is_good_line(line):
    if len(line) > 500:
        return False
    if re.search(r"\$\s|^\(base\)|Traceback|File\s\"|\.py in <module>", line):
        return False
    if re.search(r"From:.*@.*\s|Subject:|^https?://", line):
        return False
    if re.search(r"(Caltech|NASA|Ureka|Ph\.D|Postdoctoral|Technical Skills|Experience|Resume)", line):
        return False
    if re.search(r"ValueError|matplotlib|pandas|ArrowInvalid", line, re.IGNORECASE):
        return False
    if re.match(r"^\(base\)|^\$|^ls\b|^git\b|^python\b", line):
        return False
    if re.search(r"\\cite|\{.*@.*\}|\bibliographystyle", line):
        return False
    if re.search(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", line):
        return False
    return True

# --- Final cleanup ---
cleaned_lines = []
cleaned_timestamps = []

for line, ts in filtered_pairs:
    line = line.replace("\n", " ").strip()
    if not is_good_line(line):
        continue
    line = re.sub(r"\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}", "[REDACTED]", line)
    line = re.sub(r"\bShooby\b|\bShoubaneh\b", "[REDACTED]", line)
    cleaned_lines.append(line)
    cleaned_timestamps.append(ts)

print(f"✅ Total user messages extracted: {len(user_messages)}")
print(f"✅ After filtering: {len(filtered_pairs)}")
print(f"✅ Final polished lines: {len(cleaned_lines)}")

timestamp_datetimes = [datetime.fromtimestamp(ts) for ts in cleaned_timestamps]


In [ ]:
# Load sentence transformer
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(cleaned_lines)

# Optionally create a DataFrame for further use or visualization
df_embed = pd.DataFrame(embeddings)
df_embed["text"] = cleaned_lines
df_embed["timestamp"] = timestamp_datetimes


In [ ]:
# Calculate how many days ago
now = datetime(2025, 7, 26) # Or any other fixed date you prefer
days_ago = [(now - dt).days for dt in timestamp_datetimes]

# Reduce dimensions
reducer = umap.UMAP(n_neighbors=100, min_dist=0.9, metric='cosine', random_state=3)
embedding_2d = reducer.fit_transform(embeddings)

# Build DataFrame (make sure to use cleaned_lines, not prompts!)
df_vis = pd.DataFrame(embedding_2d, columns=["UMAP 1", "UMAP 2"])
df_vis["message"] = cleaned_lines
df_vis["days_ago"] = days_ago

# Plot
plt.figure(figsize=(12, 8))
sc = plt.scatter(df_vis["UMAP 1"], df_vis["UMAP 2"], c=df_vis["days_ago"], cmap="plasma_r", alpha=0.7)
plt.colorbar(sc, label="Days Ago")
plt.title("UMAP Projection of Your Messages")
plt.axis('off')
plt.grid(False)
plt.show()


In [ ]:
# KMeans clustering
n_clusters = 16
kmeans = KMeans(n_clusters=n_clusters, random_state=32)
labels = kmeans.fit_predict(embedding_2d)

# DataFrame
df_vis = pd.DataFrame(embedding_2d, columns=["UMAP 1", "UMAP 2"])
df_vis["message"] = cleaned_lines
df_vis["cluster"] = labels.astype(str)

# Moody glowing palette (deep, luminous tones)
neo_expressionist_palette = [
    "#F72585", "#7209B7", "#3A0CA3", "#4361EE", "#4CC9F0",  # magenta → cyan
    "#FFB703", "#FB8500", "#E63946", "#9A031E", "#5F0F40",  # hot yellows → red
    "#06D6A0", "#118AB2", "#073B4C", "#FFD6A5", "#FFADAD",  # cool aquas and pastels
    "#CDB4DB", "#FFAFCC", "#BDE0FE", "#A2D2FF", "#D9ED92"   # pale pastels
]

final_cluster_labels = {
    "0": "Interpersonal Insight<br>&<br>Reflection",
    "1": "Foundation Models<br>&<br>Deep Learning",
    "2": "Dev Tools<br>Git, W&B, and Beyond",
    "3": "Collaboration<br>&<br>Academic Networks",
    "4": "Data Visualization<br>&<br>Storytelling",
    "5": "Scientific AI<br>Super-Resolution & More",
    "6": "AI<br>&<br>Philosophy",
    "7": "Surrealism<br>Dali, Magritte & the Mind",
    "8": "Python Data I/O<br>&<br>Debugging",
    "9": "Existential Aesthetics<br>Kurosawa to Camus",
    "10": "Guides, Orders<br>&<br>How-To's",
    "11": "Astrophysics<br>From Telescopes to Dark Matter",
    "12": "Persian Literature<br>&<br>Cultural Nuance",
    "13": "Model Training<br>&<br>System Configuration",
    "14": "Proposals<br>&<br>Big-Picture Thinking",
    "15": "Writing<br>Polishing Prose & Paragraphs"
}

# --- Cluster centers & labels ---
df_labels = (
    df_vis.groupby("cluster")[["UMAP 1", "UMAP 2"]]
    .mean()
    .reset_index()
)
df_labels["label"] = df_labels["cluster"].map(final_cluster_labels)

# --- Base scatter plot ---
fig = px.scatter(
    df_vis,
    x="UMAP 1",
    y="UMAP 2",
    color="cluster",
    custom_data=["message"],
    color_discrete_sequence=neo_expressionist_palette,
    title="🌌 ShoobAI: UMAP of Thoughts",
    opacity=0.4,
    height=1000
)

fig.update_traces(
    hovertemplate="<b>%{customdata[0]}</b><extra></extra>",
    marker=dict(size=6, line=dict(width=0.5, color='rgba(255, 255, 255, 0.6)'))
)

for cluster_num, label in final_cluster_labels.items():
    cluster_points = df_vis[df_vis["cluster"] == str(cluster_num)]
    x_median = cluster_points["UMAP 1"].median()
    y_median = cluster_points["UMAP 2"].median()

    fig.add_annotation(
        x=x_median,
        y=y_median,
        text=label,
        showarrow=False,
        font=dict(size=14, color="white"),
        align="center",
        borderpad=4,
        bgcolor="rgba(0,0,0,0.5)",  # semi-transparent dark box
        opacity=0.95
    )

fig.update_layout(
    xaxis=dict(showgrid=False, zeroline=False, visible=False),
    yaxis=dict(showgrid=False, zeroline=False, visible=False),
    plot_bgcolor='black',
    paper_bgcolor='black',
    font=dict(color='white'),
    title_font=dict(size=24),
    showlegend=False
)

pio.renderers.default = "browser"
fig.show()


In [ ]:
# (optional) convex hulls per cluster if scipy is available
try:
    from scipy.spatial import ConvexHull
    SCIPY_OK = True
except Exception:
    SCIPY_OK = False

def hex_to_rgba(hex_color: str, alpha: float) -> str:
    hex_color = hex_color.lstrip("#")
    if len(hex_color) == 3:
        hex_color = "".join(ch*2 for ch in hex_color)
    r = int(hex_color[0:2], 16)
    g = int(hex_color[2:4], 16)
    b = int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

# -----------------------------
# KMeans clustering
n_clusters = 16
kmeans = KMeans(n_clusters=n_clusters, random_state=32)
labels = kmeans.fit_predict(embedding_2d)

# DataFrame
df_vis = pd.DataFrame(embedding_2d, columns=["UMAP 1", "UMAP 2"])
df_vis["message"] = cleaned_lines
df_vis["cluster"] = labels.astype(str)

# Moody glowing palette (deep, luminous tones)
neo_expressionist_palette = [
    "#F72585", "#7209B7", "#3A0CA3", "#4361EE", "#4CC9F0",  # magenta → cyan
    "#FFB703", "#FB8500", "#E63946", "#9A031E", "#5F0F40",  # hot yellows → red
    "#06D6A0", "#118AB2", "#073B4C", "#FFD6A5", "#FFADAD",  # cool aquas and pastels
    "#CDB4DB", "#FFAFCC", "#BDE0FE", "#A2D2FF", "#D9ED92"   # pale pastels
]

# deterministic color map per cluster
color_map = {str(i): neo_expressionist_palette[i % len(neo_expressionist_palette)]
             for i in range(n_clusters)}

final_cluster_labels = {
    "0": "Interpersonal Insight<br>&<br>Reflection",
    "1": "Foundation Models<br>&<br>Deep Learning",
    "2": "Dev Tools<br>Git, W&B, and Beyond",
    "3": "Collaboration<br>&<br>Academic Networks",
    "4": "Data Visualization<br>&<br>Storytelling",
    "5": "Scientific AI<br>Super-Resolution & More",
    "6": "AI<br>&<br>Philosophy",
    "7": "Surrealism<br>Dali, Magritte & the Mind",
    "8": "Python Data I/O<br>&<br>Debugging",
    "9": "Existential Aesthetics<br>Kurosawa to Camus",
    "10": "Guides, Orders<br>&<br>How-To's",
    "11": "Astrophysics<br>From Telescopes to Dark Matter",
    "12": "Persian Literature<br>&<br>Cultural Nuance",
    "13": "Model Training<br>&<br>System Configuration",
    "14": "Proposals<br>&<br>Big-Picture Thinking",
    "15": "Writing<br>Polishing Prose & Paragraphs"
}

# (NEW) optional: link cluster labels to subpages (edit slugs/URLs as you like)
subpage_url = {
    "0": "/people-reflection",
    "1": "/foundation-models",
    "2": "/dev-tools",
    "3": "/collaboration",
    "4": "/viz",
    "5": "/scientific-ai",
    "6": "/ai-philosophy",
    "7": "/surreal-visuals",
    "8": "/python-io-debug",
    "9": "/aesthetics-kurosawa-camus",
    "10": "/howtos",
    "11": "/astrophysics",
    "12": "/persian-culture",
    "13": "/training-config",
    "14": "/proposals-ideas",
    "15": "/writing"
}

# turn the annotation text into clickable <a> links
def make_label_html(c):
    label = final_cluster_labels[c]
    url = subpage_url.get(c, "#")
    return f"<a href='{url}' style='color:white; text-decoration:none;'>{label}</a>"

# --- Cluster centers & labels ---
df_labels = (
    df_vis.groupby("cluster")[["UMAP 1", "UMAP 2"]]
    .median()  # median is usually stabler for cluster centers on UMAP
    .reset_index()
)
df_labels["label_html"] = df_labels["cluster"].map(make_label_html)

# --- Base scatter plot (main layer) ---
fig = px.scatter(
    df_vis,
    x="UMAP 1",
    y="UMAP 2",
    color="cluster",
    custom_data=["message", "cluster"],
    color_discrete_map=color_map,
    title="🌌 ShoobAI: UMAP of Thoughts",
    opacity=0.45,
    height=1000,
    render_mode="webgl"  # smoother on large point sets
)

fig.update_traces(
    hovertemplate="<b>%{customdata[0]}</b><extra></extra>",
    marker=dict(size=6, line=dict(width=0.5, color='rgba(255,255,255,0.5)'))
)

# --- (NEW) soft glow/halo per cluster (duplicate traces, bigger size, low alpha) ---
for c in sorted(df_vis["cluster"].unique(), key=lambda x: int(x)):
    pts = df_vis[df_vis["cluster"] == c]
    fig.add_trace(
        go.Scattergl(
            x=pts["UMAP 1"],
            y=pts["UMAP 2"],
            mode="markers",
            showlegend=False,
            hoverinfo="skip",
            marker=dict(
                size=16,  # halo size
                color=color_map[c],
                opacity=0.07  # low alpha for glow
            )
        )
    )

if SCIPY_OK:
    hull_alpha = 0.10  # translucency of the “continent”
    line_alpha = 0.8   # outline alpha

    for c in sorted(df_vis["cluster"].unique(), key=lambda x: int(x)):
        pts = df_vis[df_vis["cluster"] == c][["UMAP 1", "UMAP 2"]].values
        # need >=3 non-duplicate points for a hull
        if len(pts) >= 3 and len(np.unique(pts, axis=0)) >= 3:
            try:
                hull = ConvexHull(pts)
                poly = pts[hull.vertices]
                fig.add_trace(
                    go.Scattergl(
                        x=np.append(poly[:, 0], poly[0, 0]),
                        y=np.append(poly[:, 1], poly[0, 1]),
                        mode="lines",
                        line=dict(
                            width=1,
                            color=hex_to_rgba(color_map[c].lstrip("#"), line_alpha)
                        ),
                        fill="toself",
                        fillcolor=hex_to_rgba(color_map[c], hull_alpha),
                        hoverinfo="skip",
                        showlegend=False
                    )
                )
            except Exception:
                # hull can still fail for degenerate configurations; skip quietly
                pass


# --- (NEW) starfield background (tiny, faint, random points) ---
rng = np.random.default_rng(42)
x_min, x_max = df_vis["UMAP 1"].min(), df_vis["UMAP 1"].max()
y_min, y_max = df_vis["UMAP 2"].min(), df_vis["UMAP 2"].max()

stars_n = 1500
stars_x = rng.uniform(x_min, x_max, stars_n)
stars_y = rng.uniform(y_min, y_max, stars_n)

fig.add_trace(
    go.Scattergl(
        x=stars_x, y=stars_y, mode="markers",
        marker=dict(size=2, color="rgba(255,255,255,0.15)"),
        hoverinfo="skip",
        showlegend=False
    )
)

# --- Annotations (centered on cluster medians, now clickable) ---
for _, row in df_labels.iterrows():
    c = row["cluster"]
    fig.add_annotation(
        x=row["UMAP 1"],
        y=row["UMAP 2"],
        text=row["label_html"],  # clickable <a>
        showarrow=False,
        font=dict(size=14, color="white"),
        align="center",
        borderpad=6,
        bgcolor="rgba(0,0,0,0.55)",  # semi-transparent glass
        opacity=0.98
    )

# --- Layout polish ---
fig.update_layout(
    xaxis=dict(showgrid=False, zeroline=False, visible=False),
    yaxis=dict(showgrid=False, zeroline=False, visible=False),
    plot_bgcolor='black',
    paper_bgcolor='black',
    font=dict(color='white'),
    title_font=dict(size=26),
    showlegend=False,
    hoverlabel=dict(
        bgcolor="rgba(0,0,0,0.8)",
        font_size=12,
        font_family="Helvetica"
    ),
    margin=dict(l=20, r=20, t=60, b=20)
)

# Slight vignette via margins + starfield does enough "shine" without gimmicks.
pio.renderers.default = "browser"
fig.show()


In [ ]:
# --- helpers & color map ---
def hex_to_rgba(hex_color: str, alpha: float) -> str:
    h = hex_color.lstrip("#")
    if len(h) == 3: h = "".join(ch*2 for ch in h)
    r, g, b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    return f"rgba({r},{g},{b},{alpha})"

# deterministic cluster -> color
n_clusters = 16
color_map = {str(i): neo_expressionist_palette[i % len(neo_expressionist_palette)]
             for i in range(n_clusters)}

# clickable label HTML (optional; keep plain `label` if you don't want links)
subpage_url = {
    "0": "/people-reflection", "1": "/foundation-models", "2": "/dev-tools",
    "3": "/collaboration", "4": "/viz", "5": "/scientific-ai",
    "6": "/ai-philosophy", "7": "/surreal-visuals", "8": "/python-io-debug",
    "9": "/aesthetics-kurosawa-camus", "10": "/howtos", "11": "/astrophysics",
    "12": "/persian-culture", "13": "/training-config", "14": "/proposals-ideas",
    "15": "/writing"
}
def make_label_html(c): return f"<a href='{subpage_url.get(c, '#')}' style='color:white;text-decoration:none;'>{final_cluster_labels[c]}</a>"

# --- safe hover text ---
df_vis["safe_hover"] = [msg if i in safe_message_indices else "" for i, msg in enumerate(df_vis["message"])]

# --- split & per-point colors ---
safe_df = df_vis[df_vis["safe_hover"] != ""].copy()
redacted_df = df_vis[df_vis["safe_hover"] == ""].copy()
safe_df["color"] = safe_df["cluster"].map(color_map)
redacted_df["color"] = redacted_df["cluster"].map(color_map)

# --- initialize ---
fig = go.Figure()

# --- Safe (visible + hoverable) layer ---
fig.add_trace(go.Scattergl(
    x=safe_df["UMAP 1"], y=safe_df["UMAP 2"],
    mode="markers",
    marker=dict(
        color=safe_df["color"],           # <- direct hex list (categorical)
        size=7, opacity=0.75,
        line=dict(width=0.5, color="rgba(255,255,255,0.6)")
    ),
    text=safe_df["safe_hover"],
    hovertemplate="<b>%{text}</b><extra></extra>",
    name="Safe",
    showlegend=False
))

# --- Redacted (dim, not hoverable) layer ---
fig.add_trace(go.Scattergl(
    x=redacted_df["UMAP 1"], y=redacted_df["UMAP 2"],
    mode="markers",
    marker=dict(
        color=redacted_df["color"],       # same hue, dimmer via trace opacity
        size=6
    ),
    opacity=0.30,
    hoverinfo="skip",
    name="Redacted",
    showlegend=False
))

# --- OPTIONAL: soft halos for both layers (pretty!) ---
for df in (safe_df, redacted_df):
    fig.add_trace(go.Scattergl(
        x=df["UMAP 1"], y=df["UMAP 2"],
        mode="markers",
        marker=dict(
            color=df["color"],
            size=16
        ),
        opacity=0.08, hoverinfo="skip", showlegend=False
    ))

# --- OPTIONAL: hulls with outlines (comment out to remove lines) ---
try:
    from scipy.spatial import ConvexHull
    hull_alpha, line_alpha = 0.10, 0.60
    for c in sorted(df_vis["cluster"].unique(), key=lambda x: int(x)):
        pts = df_vis[df_vis["cluster"] == c][["UMAP 1", "UMAP 2"]].values
        if len(pts) >= 3 and len(np.unique(pts, axis=0)) >= 3:
            hull = ConvexHull(pts)
            poly = pts[hull.vertices]
            fig.add_trace(go.Scattergl(
                x=np.append(poly[:,0], poly[0,0]),
                y=np.append(poly[:,1], poly[0,1]),
                mode="none",                                      # outline ON
                line=dict(width=1.5, color=hex_to_rgba(color_map[c], line_alpha)),
                fill="toself",
                fillcolor=hex_to_rgba(color_map[c], hull_alpha),
                hoverinfo="skip",
                showlegend=False
            ))
except Exception:
    pass

# --- Cluster labels (clickable) ---
for cluster_num, label in final_cluster_labels.items():
    pts = df_vis[df_vis["cluster"] == str(cluster_num)]
    if len(pts):
        fig.add_annotation(
            x=pts["UMAP 1"].median(),
            y=pts["UMAP 2"].median(),
            text=make_label_html(cluster_num),  # or just `label`
            showarrow=False,
            font=dict(size=14, color="white"),
            align="center",
            borderpad=4,
            bgcolor="rgba(0,0,0,0.55)",
            opacity=0.98
        )

# --- Layout ---
fig.update_layout(
    plot_bgcolor="black", paper_bgcolor="black",
    font=dict(color="white"), showlegend=False, height=1000,
    xaxis=dict(visible=False), yaxis=dict(visible=False),
)

# Save
pio.write_html(fig, "index.html")


In [ ]:
# Save all messages with cluster numbers for manual review
output_file = "messages_with_clusters.txt"

with open(output_file, "w", encoding="utf-8") as f:
    for idx, row in df_vis.iterrows():
        cluster = row["cluster"]
        message = row["message"].replace("\n", " ").strip()
        f.write(f"{idx}\t{message}\n")

print(f"✅ Messages saved to {output_file}")


In [ ]:
# Save all 16 clusters to a text file
output_path = "umap_clusters.txt"

with open(output_path, "w", encoding="utf-8") as f:
    for cluster_id in sorted(df_vis["cluster"].unique(), key=int):  # ensures cluster numbers are ordered
        f.write(f"\n=== Cluster {cluster_id} ===\n")
        cluster_msgs = df_vis[df_vis["cluster"] == cluster_id]["message"].tolist()
        for msg in cluster_msgs[50:100]:
            f.write(f"{msg}\n")
